# Assigning Plate IDs to XY Data

When working with paleogeographic reconstructions, it's common to have a list of modern-day coordinates (longitude, latitude). For example, fossil find locations, volcanic hotspots, or sampling stations.

If we want to **reconstruct these points to past geological times**, we first need to know:

1. Which tectonic plate each point belongs to.
2. The valid time range for that point’s position.

This tutorial shows how to:
- Read a set of points (XY data)
- Assign plate IDs and valid time periods to those points.
- Save the results into a GPML file that you can open in `pyGPlates`.

Let's begin by importing the required packages!

In [17]:
# Ensure you have the required packages installed:
# Run `pip install pygplates gplately` in your terminal if you haven't done so already.

from plate_model_manager import PlateModelManager
import pygplates

We need two things before we get started:

- A rotation model → tells the software how plates have moved over geological time.

- Static polygons → tells it which plate each location belongs to today (present-day geometry).

Without these two, the `GPlately` has no idea how your locations relate to plates, or how to move them backwards in time.

> NOTE: Using `PlateModelManager` means you don’t need to hardcode file paths, but you can also manually specify the .rot & .gpml filepaths if needed!

In [18]:
# Create a Plate Model Manager
pm_manager = PlateModelManager()

# Load & download model data into the path specified by 'data_dir'
# To load a specific model, you can specify the model name
muller2019_model = pm_manager.get_model("Muller2019", data_dir="../../data/plate-model-repo")

# Get the rotation model and static polygons from the loaded model
# Alternatively, specify the paths directly if you have them
rotation_model = muller2019_model.get_rotation_model()
static_polygons_filename = muller2019_model.get_static_polygons()

# Define output filename
output_points_filename = '../../data/cities_with_plate_ids.gpml'

Next, we create a Python list of dictionaries, where each dictionary stores:

- name → the name of the city (this will be useful for identifying it later in our results).

- lon → longitude in decimal degrees.

- lat → latitude in decimal degrees.

> NOTE: You can also import this data from a file (eg. .csv or .txt)!

After defining our XY data, we loop through each city and:

- Create a `PointOnSphere` i.e. pyGPlates’ internal representation of a geographic point.

- Create a Feature i.e. a pyGPlates object that can store geometry plus metadata.

- We attach the point geometry & the city name.

- Add the feature to a list (`point_features`).

This essentially transforms our simple Python data into something pyGPlates can understand and process.

In [19]:
# Define XY data for cities
# These are just example coordinates; you can replace them with your own data.
cities = [
    {'name': 'Sydney', 'lon': 151.2093, 'lat': -33.8688},
    {'name': 'London', 'lon': -0.1278, 'lat': 51.5074},
    {'name': 'Tokyo', 'lon': 139.6917, 'lat': 35.6895},
    {'name': 'Cairo', 'lon': 31.2357, 'lat': 30.0444},
    {'name': 'Rio de Janeiro', 'lon': -43.1729, 'lat': -22.9068},
    {'name': 'New York', 'lon': -74.0060, 'lat': 40.7128}
]

point_features = []

for city in cities:
    point = pygplates.PointOnSphere(city['lat'], city['lon'])  # lat/lon order for pyGPlates
    feature = pygplates.Feature()
    feature.set_geometry(point)

    # Store the city name as a property so it’s preserved in GPML
    feature.set_name(city['name'])

    # Add the feature to the list
    point_features.append(feature)

Now, we assign plate IDs to these points! Here's how it works:

- The function compares each city’s location (`point_features`) with the static polygons (present-day plate boundaries).

- It uses the rotation model to understand how these polygons relate to plates in the reconstruction.

- `properties_to_copy` tells pyGPlates which extra attributes to attach to each feature — in this case, both the plate ID and valid time period.

The result is a new list of features (`assigned_features`) with plate information attached, ready for reconstruction or export.

In [20]:
# Use the static polygons to assign plate IDs and valid time periods.
# Each point feature is partitioned into one of the static polygons and assigned its
# reconstruction plate ID and valid time period.

assigned_features = pygplates.partition_into_plates(
    static_polygons_filename,
    rotation_model,
    point_features,
    properties_to_copy=[
        pygplates.PartitionProperty.reconstruction_plate_id,
        pygplates.PartitionProperty.valid_time_period
    ]
)

Next, we:

- Wrap the assigned features into a `FeatureCollection` (this is pyGPlates’ container for storing multiple features together).

- Write the collection to a `GPML` file using .write(). GPML (GPlates Markup Language) is the format GPlates uses to store reconstructed geometries and metadata.

Once saved, you can open this .gpml file in `GPlately` to visualise the cities with their assigned plate IDs and time periods!

In [21]:
# Save the assigned features to a new GPML file
feature_collection = pygplates.FeatureCollection(assigned_features)
feature_collection.write(output_points_filename)

print(f"Saved results to {output_points_filename}")

Saved results to ../../data/cities_with_plate_ids.gpml
